In [2]:
from pydantic import BaseModel, field_validator
from typing import Optional, List, Dict, Any
from datetime import datetime
import re


class Document(BaseModel):
    _id: str
    pid: str
    title: str
    description: Optional[str] = None
    brand: Optional[str] = None
    category: Optional[str] = None
    sub_category: Optional[str] = None
    product_details: Optional[Dict[str, Any]] = None
    seller: Optional[str] = None
    out_of_stock: bool = False
    selling_price: Optional[float] = None
    discount: Optional[float] = None
    actual_price: Optional[float] = None
    average_rating: Optional[float] = None
    url: Optional[str] = None
    images: Optional[List[str]] = None

    def to_json(self):
        return self.model_dump_json()

    # --- Validators ---

    @field_validator("selling_price", "actual_price", mode="before")
    def parse_price(cls, v):
        if v is None:
            return None
        if isinstance(v, str):
            v = v.strip().replace(",", "")
            if v == "":
                return None
            try:
                return float(v)
            except ValueError:
                return None
        return v

    @field_validator("average_rating", mode="before")
    def parse_rating(cls, v):
        if v is None:
            return None
        if isinstance(v, str):
            v = v.strip()
            if v == "":
                return None
            try:
                return float(v)
            except ValueError:
                return None
        return v

    @field_validator("discount", mode="before")
    def parse_discount(cls, v):
        if v is None:
            return None
        if isinstance(v, str):
            match = re.search(r"(\d+(?:\.\d+)?)", v.replace(",", ""))
            if match:
                return float(match.group(1))
            return None
        return v

    @field_validator("product_details", mode="before")
    def normalize_product_details(cls, v):
        if isinstance(v, list):
            merged = {}
            for item in v:
                if isinstance(item, dict):
                    merged.update(item)
            return merged
        return v

    def __str__(self) -> str:
        return self.model_dump_json(indent=2)


class StatsDocument(BaseModel):
    """
    Original corpus data as an object
    """
    pid: str
    title: str
    description: Optional[str] = None
    url: Optional[str] = None
    count: Optional[int] = None

    def __str__(self) -> str:
        return self.model_dump_json(indent=2)
    
    def to_json(self):
        return self.model_dump_json()


class ResultItem(BaseModel):
    pid: str
    title: str
    description: Optional[str] = None
    url: Optional[str] = None
    ranking: Optional[float] = None

    def __str__(self) -> str:
        return self.model_dump_json(indent=2)
    
    def to_json(self):
        return self.model_dump_json()


In [3]:
# Preparing environment

import os
import sys
import random
import string
import json
import math
import numpy as np
import pandas as pd

from typing import List, Dict

import  nltk
nltk.download('stopwords')

from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from collections import defaultdict, Counter
from array import array

from dotenv import load_dotenv

load_dotenv()  # take environment variables from .env

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/estudiant/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
def build_terms(text):
    """
    Preprocesses the text fields of the document in the corpus (only`title` and `description`) by removing stop words, tokenizing, removing punctuation marks, stemming and [#TODO].

    :param text: (string) text to be processed
    :return text: List of tokens corresponding to the input text after the preprocessing
    """

    stemmer = PorterStemmer()
    stop_words = set(stopwords.words("english"))

    text = text.lower()  # Transform to lowercase
    text = text.split(" ")  # Tokenize the text, separating by spaces
    text = [
        word.strip(string.punctuation)
        for word in text
        if word.strip(string.punctuation).isalnum() and word not in stop_words
    ]  # eliminate the stop words [`strip` is to separate exclamation signs from words, e.g. "Hello!"" -> "Hello" + "!"]
    text = [stemmer.stem(word) for word in text]  # perform stemming

    # TODO: add more preprocessing if necessary

    return text

def join_build_terms(strings):
    """
    Builds the terms by concatenating the strings in the given list.

    :param strings: List of string. Texts to be concatenated and processed.
    :return terms: List of string, where each item is a word.
    """
    arg = " ".join(strings)
    return build_terms(arg)

In [5]:
def _build_corpus(df: pd.DataFrame) -> Dict[str, Document]:
    """
    Build corpus from dataframe
    :param df:
    :return:
    """
    corpus = {}
    for _, row in df.iterrows():
        doc = Document(**row.to_dict())
        corpus[doc.pid] = doc
    return corpus


def load_corpus(path) -> List[Document]:
    """
    Load file and transform to dictionary with each document as an object for easier treatment when needed for displaying
     in results, stats, etc.
    :param path:
    :return:
    """
    df = pd.read_json(path)
    corpus = _build_corpus(df)
    return corpus


def corpus_df_loading(path):
    """
    In this function we load the corpus as dataframe, and we preprocess the numerical fields.

    :param path: Path to the json file.
    :return corpus: Returns a dictionary List[Document] with the loaded corpus with the numerical fields preprocessed.
    """
    corpus = load_corpus(path)
    return corpus

In [6]:
def create_index_tf_idf(corpus):
    """
    Implement the inverted index and compute tf, df and idf

    Argument:
    corpus -- 

    #TODO: adapt to our version

    Returns:
    index - the inverted index (implemented through a Python dictionary) containing terms as keys and the corresponding
    list of document these keys appears in (and the positions) as values.
    index2title - a mapping of article pid to its title
    tf - normalized term frequency for each term in each document
    df - number of documents each term appear in
    idf - inverse document frequency of each term
    """

    index = defaultdict(dict)
    tf = defaultdict(dict)
    df = defaultdict(dict)
    idf = defaultdict(dict)
    index2title = {}
    num_articles = len(corpus)

    for doc in list(corpus.values()):
        index2title[doc.pid] = doc.title
        # For each field to be considered in the index, get its terms
        title_description = join_build_terms([doc.title, doc.description])  # Pre-process the `title` and `description`
        brand_terms = join_build_terms([doc.brand])
        category_terms = join_build_terms([doc.category])
        sub_category_terms = join_build_terms([doc.sub_category])
        seller_product_details = join_build_terms([doc.seller, " ".join([detail for detail in doc.product_details.values()])])  # Pre-process the `title` and `description`

        # Fields and target terms to process
        fields = ['title_description', 'brand', 'category', 'sub_category', 'seller_product_details']
        target = [title_description, brand_terms, category_terms, sub_category_terms, seller_product_details]

        # Initialize a temporal dictionary to store the index terms for the current article
        current_article_index = defaultdict(dict)

        # Create the index for the current article
        # For each field we consider in the index
        for idx, field in enumerate(fields):    
            # For each term in the target field
            for position, term in enumerate(target[idx]):   
                try:
                    # Add the new found term's position to the dict
                    current_article_index[term][field][1].append(position)  
                except:
                    # Create the entry for the term and field with the term's position if it didn't exist
                    current_article_index[term][field] = [doc.pid, [position]]  

        for field in fields:
            norm = 0
            for term, positions in current_article_index.items():
                norm += len(positions[field][1])**2
            norm = math.sqrt(norm)

            for term, positions in current_article_index.items():
                # Compute term frequency and document frequency of each term per category
                try:
                    tf[term][field].append(np.round(len(positions[field][1])/norm, 4))
                    df[term][field] += 1
                # If it's a term we haven't seen, create a new term frequency and document frequency entry
                except:
                    tf[term][field] = [np.round(len(positions[field][1])/norm, 4)]
                    df[term][field] = 1

                # In the practice, the tf/df and the index were in separate loops. Both codes are now in one 
                # loop to avoid reading the same twice
                # Join the current article's index with the global index
                try:
                    index[term][field].append(positions[field])  # Add the array of positions ("[id, [[0],[1]]]"") in the given term and field
                except:
                    index[term][field] = [positions[field]]      # Create the entry for the term and the field with the array of positions


    for term, posting_fields in df:
        for field, posting in posting_fields:
            idf[term][field] = np.round(np.log(float(num_articles / df[term])), 4)

    return index, index2title, tf, df, idf

In [7]:
json_path = "../../data/fashion_products_dataset.json"
corpus = corpus_df_loading(json_path)

In [8]:
print(corpus.keys())

dict_keys(['TKPFCZ9EA7H5FYZH', 'TKPFCZ9EJZV2UVRZ', 'TKPFCZ9EHFCY5Z4Y', 'TKPFCZ9ESZZ7YWEF', 'TKPFCZ9EVXKBSUD7', 'TKPFCZ9EFK9DNWDA', 'TKPFDABN3GXYPFHE', 'TKPFCZ9ESGZYT8NH', 'TKPFCZ9DYU33FFXS', 'TKPFDABN4NQFVKZY', 'TKPFCZ9ENWGMX23W', 'TKPFZFSHHACG3FHC', 'TKPFZFSHQPDRGZTM', 'TKPFZ4YTRF3ZRTTH', 'TKPFZ4YTJZWBFYFZ', 'TKPFZFSH3F9ZA7C6', 'TKPFZ4YT7ZNYXG27', 'TKPFZ4YTGNZJDZDU', 'TKPFZ4YTX94CY9JX', 'TKPFCZ9EHCNAPKPU', 'TKPFDACEXAWUHGR7', 'TKPFCZ9ETR6YVXNG', 'TKPFD3K6K5TNYZGF', 'TKPFCZ9EGGYENTZS', 'TKPFD3K6ZMN79MPH', 'TKPFD3K6UZBYDZNY', 'TKPFD3K62JB9PEMR', 'TKPFCZ9EZDPZR5AH', 'TKPFWBGVGU9FCAYX', 'TKPFWBGPGVKVZARU', 'TKPFCZ9EVM2GZ4GF', 'TKPFWAG7YFWPMG5Y', 'TKPFCZ9E2UC3DR3F', 'TKPFCZ9ECDYYDNKA', 'SOCFVCCYUDEN2AVH', 'CTPFVZTEMJWEJJJV', 'CTPFVQP2PEHKGYFQ', 'CTPFVPM3NDPBPPXE', 'CTPFVZHSA7G4PFC5', 'CTPFVZD8CNSZ3AMR', 'CTPFVQNNHGYFTGFN', 'CTPFVZT3UFN99ZTH', 'CTPFVZHY42MSZCF6', 'CTPFVSU7CXFCXEHD', 'CTPFVZGRKPGSFPUU', 'CTPFVZT7EFZWVRUP', 'CTPFVXGGEHH6YY8G', 'CTPFVQZFE4HTFZNG', 'CTPFVPKJGJBXHQFJ', 'CTPFVZT2

In [9]:
def create_index_tf_idf(corpus):
    """
    Implement the inverted index and compute tf, df and idf

    Argument:
    corpus --

    #TODO: adapt to our version

    Returns:
    index - the inverted index (implemented through a Python dictionary) containing terms as keys and the corresponding
    list of document these keys appears in (and the positions) as values.
    index2title - a mapping of article pid to its title
    tf - normalized term frequency for each term in each document
    df - number of documents each term appear in
    idf - inverse document frequency of each term
    """

    index = defaultdict(dict)
    tf = defaultdict(lambda: defaultdict(dict))
    df = defaultdict(int)
    idf = {}
    index2title = {}
    num_articles = len(corpus)

    for product in list(corpus.values()):
        index2title[product.pid] = product.title
        # For each field to be considered in the index, get its terms
        title_description = join_build_terms(
            [product.title, product.description]
        )  # Pre-process the `title` and `description`
        brand_terms = join_build_terms([product.brand])
        category_terms = join_build_terms([product.category])
        sub_category_terms = join_build_terms([product.sub_category])
        seller_product_details = join_build_terms(
            [product.seller, " ".join([detail for detail in product.product_details.values()])]
        )  # Pre-process the `title` and `description`

        # Fields and target terms to process
        fields = [
            "title_description",
            "brand",
            "category",
            "sub_category",
            "seller_product_details",
        ]
        target = [
            title_description,
            brand_terms,
            category_terms,
            sub_category_terms,
            seller_product_details,
        ]

        # Initialize a temporal dictionary to store the index terms for the current article
        current_article_index = defaultdict(dict)

        # Create the index for the current article
        # For each field we consider in the index
        for field_idx, field in enumerate(fields):
            # For each term in the target field
            for position, term in enumerate(target[field_idx]):
                try:
                    # Add the new found term's position to the dict
                    current_article_index[term][field][1].append(position)
                except:
                    # Create the entry for the term and field with the term's position if it didn't exist
                    current_article_index[term][field] = [product.pid, [position]]

        seen_terms = set()
        for field in fields:
            norm = 0
            for term, postings in current_article_index.items():
                if field in postings.keys():    # Check that the term appear in the current field
                    norm += len(postings[field][1]) ** 2
            norm = math.sqrt(norm)

            for term, postings in current_article_index.items():
                if field in postings.keys():
                    pid = postings[field][0]
                    # Compute term frequency and document frequency of each term per category
                    tf[pid][term][field] = np.round(len(postings[field][1]) / norm, 4)
                    

                    # In the practice, the tf/df and the index were in separate loops. Both codes are now in one
                    # loop to avoid reading the same twice
                    # Join the current article's index with the global index
                    try:
                        index[term][field].append(
                            postings[field]
                        )  # Add the array of positions ("[id, [[0],[1]]]"") in the given term and field
                    except:
                        index[term][field] = [
                            postings[field]
                        ]  # Create the entry for the term and the field with the array of positions

                if term not in seen_terms:
                    df[term] += 1 
                    seen_terms.add(term)
        


    for term in df.keys():
        idf[term] = (
            np.round(np.log(float(num_articles / df[term])), 4)
        )

    return index, index2title, tf, df, idf


In [10]:
index, index2title, tf, df, idf = create_index_tf_idf(corpus=corpus)

In [11]:
def search(query, index):
    """
    The output is the list of documents that contain ALL query terms.

    :param query: (string) query
    :param index: (Dict) inverted index dictinary
    :return selected_docs: (List) of documents' ids that contain all query terms
    """

    query_terms = build_terms(query)
    docs = None
    for term in query_terms:
        try:
            # Get all doc ids from the term
            all_doc_ids = [
                doc_id
                for categories in index[term].values()
                for doc_id, _ in categories
            ]

            # Get intersection of documents with ALL the terms
            if docs is None:  # First time, set is empty
                docs = set(all_doc_ids)  # Initiallize with first term's doc ids
            else:
                docs &= set(all_doc_ids)

        except:
            pass

    return query_terms, list(docs)

In [12]:
type(index2title)

dict

In [13]:
query = "slim jeans"
query_terms, products = search(query=query, index=index)

weights = {
        "title_description": 1.0,
        "brand": 1.0,
        "category": 1.0,
        "sub_category": 1.0,
        "seller_product_details": 1.0,
    } 


In [14]:
# def rank_tf_idf(query_terms, products, index, tf, idf, weights):
#     """
#     Perform the ranking of the results of a search based on the tf-idf weights

#     Argument:
#     terms -- list of query terms
#     products -- list of products to rank that match the query
#     index -- inverted index data structure
#     tf -- term frequencies
#     idf -- inverted document frequencies
#     weights -- weights to average the fields to compute the rank

#     Returns:
#     Print the list of product ids of the ranked articles
#     """
#     # For the docs, take only the components for the query terms 
product_vectors = defaultdict(lambda: [0]*len(query_terms))
query_vector = [0] * len(query_terms)


# Compute the norm for the query tf
query_term_counts = Counter(query_terms)
query_norm = np.linalg.norm(list(query_term_counts.values()))

# Compute tf-idf for each document and query
for term_idx, q_term in enumerate(query_terms):
    if q_term not in index:
        continue

    # tf*idf (normalize TF)
    query_vector[term_idx] = (query_term_counts[q_term]/query_norm) * idf[q_term]

    # Compute the document vectors
    second_loop = False
    for field, postings in index[q_term].items():
        for pid, positions in postings:
            # print(pid)
            # print(positions)
            # print(type(tf[pid][q_term][field]))
            # print(type(weights[field]))
            product_vectors[pid][term_idx] += tf[pid][q_term][field]* weights[field]
     
product_scores = [[np.dot(current_prod_vec, query_vector), prod] for prod, current_prod_vec in product_vectors.items()]
product_scores.sort(reverse=True)

# print(product_scores)

defaultdict(<function __main__.<lambda>()>,
            {'CTPFVZHSA7G4PFC5': [np.float64(0.0572), 0],
             'CTPFVZD8CNSZ3AMR': [np.float64(0.0572), 0],
             'CTPFVZT3UFN99ZTH': [np.float64(0.1231), 0]})

defaultdict(<function __main__.<lambda>()>,
            {'CTPFVZHSA7G4PFC5': [np.float64(0.0572), 0],
             'CTPFVZD8CNSZ3AMR': [np.float64(0.0572), 0],
             'CTPFVZT3UFN99ZTH': [np.float64(0.1231), 0],
             'TSHFHJKSY8FNR7PV': [0, np.float64(0.1021)],
             'TSHFHJKRPYSHNEMG': [0, np.float64(0.125)],
             'JEAFUZ87WN7FWM8A': [0, np.float64(0.3714)]})